# Notebook 02 — Neo4j Graph Load + Cypher Queries

**Graph Schema:**
```
Nodes:         Movie, Director, Actor, Producer, Writer, DOP, Composer
Relationships: DIRECTED_BY, ACTED_IN, PRODUCED_BY, WRITTEN_BY, SHOT_BY, SCORE_BY
```

**Business Question:** *Which film professionals — across ALL crew roles — are most central to the Hollywood success network, and which cross-role combinations consistently produce top-performing films?*

> **Pre-requisite:** Run `docker-compose up -d` from the project root before executing this notebook.  
> Neo4j will be available at `bolt://localhost:7687` (user: neo4j, password: capstone2024)

In [5]:
from neo4j import GraphDatabase
import pandas as pd
import os
from pathlib import Path

URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD", "capstone2024")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

def run(cypher, label=None, **params):
    with driver.session() as s:
        result = s.run(cypher, **params)
        df = pd.DataFrame([r.data() for r in result])
    if label:
        print(f"\n=== {label} ===")
        print(df.to_string(index=False))
    return df

# Connectivity check
run("RETURN 'Connected' AS status", label="Neo4j Status")


=== Neo4j Status ===
   status
Connected


,status
0,Connected


In [6]:
with driver.session() as s:
    deleted = 1
    while deleted > 0:
        deleted = s.run(
            "MATCH (n) WITH n LIMIT 10000 DETACH DELETE n RETURN count(*) AS d"
        ).single()["d"]
        print(f"Deleted {deleted} nodes...")
print("Done — now run your cell.")

Deleted 0 nodes...
Done — now run your cell.


## Step 1 — Clear and Create Constraints

In [7]:
# Clear existing data (idempotent re-run)
run("MATCH (n) DETACH DELETE n")
print("Graph cleared.")

# Uniqueness constraints
constraints = [
    "CREATE CONSTRAINT movie_id    IF NOT EXISTS FOR (m:Movie)    REQUIRE m.movie_id IS UNIQUE",
    "CREATE CONSTRAINT director_nm IF NOT EXISTS FOR (d:Director) REQUIRE d.name IS UNIQUE",
    "CREATE CONSTRAINT actor_nm    IF NOT EXISTS FOR (a:Actor)    REQUIRE a.name IS UNIQUE",
    "CREATE CONSTRAINT producer_nm IF NOT EXISTS FOR (p:Producer) REQUIRE p.name IS UNIQUE",
    "CREATE CONSTRAINT writer_nm   IF NOT EXISTS FOR (w:Writer)   REQUIRE w.name IS UNIQUE",
    "CREATE CONSTRAINT dop_nm      IF NOT EXISTS FOR (d:DOP)      REQUIRE d.name IS UNIQUE",
    "CREATE CONSTRAINT composer_nm IF NOT EXISTS FOR (c:Composer) REQUIRE c.name IS UNIQUE",
]
for c in constraints:
    run(c)
print("Constraints created.")

Graph cleared.
Constraints created.


## Step 2 — Load Node CSVs via Python Batching

> We use Python batching rather than `LOAD CSV` so we can run on any machine without placing files in the Neo4j import directory.

In [8]:
import csv

DATA_DIR = Path("../data/neo4j_csv")
BATCH = 1000

def batch_load(session, cypher_template, csv_file, transform_fn):
    """Stream CSV rows into Neo4j in batches."""
    rows, total = [], 0
    with open(csv_file, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            transformed = transform_fn(row)
            if transformed:
                rows.append(transformed)
            if len(rows) >= BATCH:
                session.run(cypher_template, rows=rows)
                total += len(rows)
                rows = []
    if rows:
        session.run(cypher_template, rows=rows)
        total += len(rows)
    return total

In [9]:
# Load Movies
def safe_float(v):
    try: return float(v) if v not in (None, '', 'None') else None
    except: return None

def safe_int(v):
    try: return int(float(v)) if v not in (None, '', 'None') else None
    except: return None

MOVIE_CQL = """
UNWIND $rows AS r
MERGE (m:Movie {movie_id: r.movie_id})
SET m.title             = r.title,
    m.release_year      = r.release_year,
    m.vote_average      = r.vote_average,
    m.vote_count        = r.vote_count,
    m.budget            = r.budget,
    m.revenue           = r.revenue,
    m.roi_pct           = r.roi_pct,
    m.popularity        = r.popularity,
    m.runtime           = r.runtime,
    m.genres            = r.genres,
    m.original_language = r.original_language,
    m.overview          = r.overview,
    m.tagline           = r.tagline,
    m.is_successful         = r.is_successful,
    m.community_vfx_score   = r.community_vfx_score,
    m.vfx_percentage        = r.vfx_percentage,
    m.community_id          = r.community_id,
    m.is_vfx                = r.is_vfx
"""

with driver.session() as s:
    n = batch_load(s, MOVIE_CQL, DATA_DIR / "nodes_Movie.csv", lambda r: {
        "movie_id":     safe_int(r.get("movie_id")),
        "title":        r.get("title"),
        "release_year": safe_int(r.get("release_year")),
        "vote_average": safe_float(r.get("vote_average")),
        "vote_count":   safe_int(r.get("vote_count")),
        "budget":       safe_float(r.get("budget")),
        "revenue":      safe_float(r.get("revenue")),
        "roi_pct":      safe_float(r.get("roi_pct")),
        "popularity":   safe_float(r.get("popularity")),
        "runtime":      safe_int(r.get("runtime")),
        "genres":       r.get("genres"),
        "original_language": r.get("original_language"),
        "overview":     r.get("overview"),
        "tagline":      r.get("tagline"),
        "is_successful":       safe_int(r.get("is_successful", 0)),
        "community_vfx_score": safe_float(r.get("community_vfx_score")),
        "vfx_percentage":      safe_float(r.get("vfx_percentage")),
        "community_id":        safe_int(r.get("community_id")),
        "is_vfx":              safe_int(r.get("is_vfx", 0))
    })
print(f"✅ Movies loaded: {n:,}")

✅ Movies loaded: 209,415


In [10]:
# Load People nodes (all crew types)
PERSON_CQL = "UNWIND $rows AS r MERGE (:{label} {{name: r.name}})"

crew_types = [
    ("Director", "nodes_Director.csv"),
    ("Actor",    "nodes_Actor.csv"),
    ("Producer", "nodes_Producer.csv"),
    ("Writer",   "nodes_Writer.csv"),
    ("DOP",      "nodes_DOP.csv"),
    ("Composer", "nodes_Composer.csv"),
]

name_fn = lambda r: {"name": r.get("name", "").strip()} if r.get("name", "").strip() else None

with driver.session() as s:
    for label, filename in crew_types:
        cql = f"UNWIND $rows AS r MERGE (:{label} {{name: r.name}})"
        n = batch_load(s, cql, DATA_DIR / filename, name_fn)
        print(f"✅ {label}: {n:,}")

✅ Director: 79,731
✅ Actor: 892,941
✅ Producer: 134,170
✅ Writer: 120,775
✅ DOP: 32,148
✅ Composer: 20,268


In [11]:
# Load Relationships
rels = [
    ("DIRECTED_BY", "rel_DIRECTED_BY.csv",  "movie_id", "director_name", "Director"),
    ("ACTED_IN",    "rel_ACTED_IN.csv",     "movie_id", "actor_name",   "Actor"),
    ("PRODUCED_BY", "rel_PRODUCED_BY.csv",  "movie_id", "producer_name","Producer"),
    ("WRITTEN_BY",  "rel_WRITTEN_BY.csv",   "movie_id", "writer_name",  "Writer"),
    ("SHOT_BY",     "rel_SHOT_BY.csv",      "movie_id", "dop_name",     "DOP"),
    ("SCORE_BY",    "rel_SCORE_BY.csv",     "movie_id", "composer_name","Composer"),
]

with driver.session() as s:
    for rel_type, csv_file, movie_col, person_col, person_label in rels:
        cql = f"""
        UNWIND $rows AS r
        MATCH (m:Movie  {{movie_id: r.movie_id}})
        MATCH (p:{person_label} {{name: r.person_name}})
        MERGE (m)-[:{rel_type}]->(p)
        """
        def make_row(row, mc=movie_col, pc=person_col):
            mid = safe_int(row.get(mc))
            pname = row.get(pc, "").strip()
            if mid and pname:
                return {"movie_id": mid, "person_name": pname}
            return None
        n = batch_load(s, cql, DATA_DIR / csv_file, make_row)
        print(f"✅ {rel_type}: {n:,}")

✅ DIRECTED_BY: 197,016
✅ ACTED_IN: 2,396,768
✅ PRODUCED_BY: 373,599
✅ WRITTEN_BY: 258,071
✅ SHOT_BY: 89,001
✅ SCORE_BY: 50,739


## Step 3 — Graph Summary

In [12]:
run("""
    MATCH (n)
    RETURN labels(n)[0] AS node_type, COUNT(n) AS count
    ORDER BY count DESC
""", label="Node Counts")


=== Node Counts ===
node_type  count
    Actor 892939
    Movie 209415
 Producer 134169
   Writer 120775
 Director  79731
      DOP  32148
 Composer  20268


,node_type,count
0,Actor,892939
1,Movie,209415
2,Producer,134169
3,Writer,120775
4,Director,79731
5,DOP,32148
6,Composer,20268


In [13]:
run("""
    MATCH ()-[r]->()
    RETURN type(r) AS rel_type, COUNT(r) AS count
    ORDER BY count DESC
""", label="Relationship Counts")


=== Relationship Counts ===
   rel_type   count
   ACTED_IN 2396768
PRODUCED_BY  373599
 WRITTEN_BY  258071
DIRECTED_BY  197016
    SHOT_BY   89001
   SCORE_BY   50739


,rel_type,count
0,ACTED_IN,2396768
1,PRODUCED_BY,373599
2,WRITTEN_BY,258071
3,DIRECTED_BY,197016
4,SHOT_BY,89001
5,SCORE_BY,50739


## Step 4 — Business Cypher Queries (10 Queries)

Each query is framed around the core business question: **who are the most impactful film industry professionals?**

In [14]:
# Q1 — Top Actors by Successful Films
run("""
    MATCH (a:Actor)<-[:ACTED_IN]-(m:Movie)
    WITH a, COUNT(m) AS total_films, SUM(m.is_successful) AS hits
    WHERE total_films >= 5
    RETURN a.name AS actor, total_films, hits,
           ROUND(100.0 * hits / total_films, 1) AS hit_rate_pct
    ORDER BY hits DESC
    LIMIT 15
""", label="Q1 — Top Actors by Hit Films")


=== Q1 — Top Actors by Hit Films ===
                   actor  total_films  hits  hit_rate_pct
            Frank Welker          219    83          37.9
            Grey DeLisle          189    77          40.7
         Fred Tatasciore          154    50          32.5
             Tara Strong          163    47          28.8
       Dee Bradley Baker          176    45          25.6
Kevin Michael Richardson          144    43          29.9
        Kappei Yamaguchi          110    39          35.5
         Koichi Yamadera          147    37          25.2
           John DiMaggio          111    34          30.6
              Ikue Otani          103    34          33.0
               Yuki Kaji           89    34          38.2
             Akio Otsuka          107    32          29.9
          Maaya Sakamoto           91    32          35.2
            Jeff Bennett          144    31          21.5
        Takahiro Sakurai          113    31          27.4


,actor,total_films,hits,hit_rate_pct
0,Frank Welker,219,83,37.9
1,Grey DeLisle,189,77,40.7
2,Fred Tatasciore,154,50,32.5
3,Tara Strong,163,47,28.8
4,Dee Bradley Baker,176,45,25.6
5,Kevin Michael Richardson,144,43,29.9
6,Kappei Yamaguchi,110,39,35.5
7,Koichi Yamadera,147,37,25.2
8,John DiMaggio,111,34,30.6
9,Ikue Otani,103,34,33.0


In [15]:
# Q2 — Top Directors by Average ROI
run("""
    MATCH (m:Movie)-[:DIRECTED_BY]->(d:Director)
    WHERE m.roi_pct IS NOT NULL
    WITH d, COUNT(m) AS films, AVG(m.roi_pct) AS avg_roi
    WHERE films >= 3
    RETURN d.name AS director, films,
           ROUND(avg_roi, 1) AS avg_roi_pct
    ORDER BY avg_roi_pct DESC
    LIMIT 15
""", label="Q2 — Top Directors by Average ROI")


=== Q2 — Top Directors by Average ROI ===
                   director  films  avg_roi_pct
                 Ron Howard     14   17548395.6
              Kimo Stamboel      3     299956.5
                    P. Vasu      3     116603.4
            Ram Babu Gurung      4      69583.7
              Roman Karimov      4      25446.6
             Haruo Sotozaki      3       9625.7
                John Carney      3       4592.8
Ariel Schulman, Henry Joost      4       4299.7
               Damien Leone      3       3906.0
              Alex Kendrick      7       3438.2
            Gianluca Leuzzi      4       3150.9
                 Jared Hess      5       2414.0
                 Joko Anwar      5       2406.8
                 Joel Zwick      3       2394.7
               Jordan Peele      3       2303.0


,director,films,avg_roi_pct
0,Ron Howard,14,17548395.6
1,Kimo Stamboel,3,299956.5
2,P. Vasu,3,116603.4
3,Ram Babu Gurung,4,69583.7
4,Roman Karimov,4,25446.6
5,Haruo Sotozaki,3,9625.7
6,John Carney,3,4592.8
7,"Ariel Schulman, Henry Joost",4,4299.7
8,Damien Leone,3,3906.0
9,Alex Kendrick,7,3438.2


In [16]:
# Q3 — Top Producers by Portfolio Revenue
run("""
    MATCH (m:Movie)-[:PRODUCED_BY]->(p:Producer)
    WHERE m.revenue IS NOT NULL
    WITH p, COUNT(m) AS films, SUM(m.revenue) AS total_revenue, AVG(m.vote_average) AS avg_rating
    WHERE films >= 3
    RETURN p.name AS producer, films,
           ROUND(total_revenue / 1e9, 2) AS total_revenue_bn_usd,
           ROUND(avg_rating, 2) AS avg_rating
    ORDER BY total_revenue DESC
    LIMIT 15
""", label="Q3 — Top Producers by Revenue Portfolio")


=== Q3 — Top Producers by Revenue Portfolio ===
          producer  films  total_revenue_bn_usd  avg_rating
       Kevin Feige     48                 36.35        6.96
          Stan Lee     58                 35.96        6.75
  Louis D'Esposito     38                 32.51        7.17
   Victoria Alonso     29                 27.61        7.29
  Steven Spielberg     50                 18.59        6.85
       Thomas Tull     51                 18.24        6.68
     John Lasseter     30                 17.53        7.31
          Avi Arad     38                 17.44        6.50
      Bruce Berman     90                 16.64        6.41
     Toby Emmerich     96                 15.89        6.36
      David Heyman     27                 14.59        7.24
  Chris Meledandri     21                 13.56        6.94
       Jon Favreau     13                 13.06        7.16
Jeffrey Katzenberg     31                 12.83        6.83
    Neal H. Moritz     55                 12.83    

,producer,films,total_revenue_bn_usd,avg_rating
0,Kevin Feige,48,36.35,6.96
1,Stan Lee,58,35.96,6.75
2,Louis D'Esposito,38,32.51,7.17
3,Victoria Alonso,29,27.61,7.29
4,Steven Spielberg,50,18.59,6.85
5,Thomas Tull,51,18.24,6.68
6,John Lasseter,30,17.53,7.31
7,Avi Arad,38,17.44,6.50
8,Bruce Berman,90,16.64,6.41
9,Toby Emmerich,96,15.89,6.36


In [17]:
# Q4 — Top Writers by Critical Success Rate
run("""
    MATCH (m:Movie)-[:WRITTEN_BY]->(w:Writer)
    WITH w, COUNT(m) AS films, SUM(m.is_successful) AS hits, AVG(m.vote_average) AS avg_rating
    WHERE films >= 4
    RETURN w.name AS writer, films, hits,
           ROUND(avg_rating, 2) AS avg_rating,
           ROUND(100.0 * hits / films, 1) AS hit_rate_pct
    ORDER BY hits DESC
    LIMIT 15
""", label="Q4 — Top Writers by Critical Success")


=== Q4 — Top Writers by Critical Success ===
                 writer  films  hits  avg_rating  hit_rate_pct
               Bob Kane     60    36        7.03          60.0
             Jack Kirby     54    27        6.93          50.0
               Stan Lee     52    23        6.70          44.2
            Bill Finger     37    21        7.00          56.8
           Ruth Handler     37    20        7.01          54.1
           Jerry Siegel     42    20        6.88          47.6
            Joe Shuster     42    20        6.88          47.6
            Lee Yoon-ho     32    19        7.02          59.4
            Mark Monroe     51    17        6.63          33.3
William Moulton Marston     29    15        6.94          51.7
           Eiichiro Oda     25    14        6.95          56.0
          Reiko Yoshida     39    14        6.74          35.9
   Anders Thomas Jensen     42    12        6.38          28.6
            Steve Ditko     20    12        7.03          60.0
   Amitab

,writer,films,hits,avg_rating,hit_rate_pct
0,Bob Kane,60,36,7.03,60.0
1,Jack Kirby,54,27,6.93,50.0
2,Stan Lee,52,23,6.70,44.2
3,Bill Finger,37,21,7.00,56.8
4,Ruth Handler,37,20,7.01,54.1
5,Jerry Siegel,42,20,6.88,47.6
6,Joe Shuster,42,20,6.88,47.6
7,Lee Yoon-ho,32,19,7.02,59.4
8,Mark Monroe,51,17,6.63,33.3
9,William Moulton Marston,29,15,6.94,51.7


In [18]:
# Q5 — Top DOPs by Avg Rating
run("""
    MATCH (m:Movie)-[:SHOT_BY]->(d:DOP)
    WITH d, COUNT(m) AS films, AVG(m.vote_average) AS avg_rating, SUM(m.revenue) AS total_rev
    WHERE films >= 5
    RETURN d.name AS cinematographer, films,
           ROUND(avg_rating, 2) AS avg_rating,
           ROUND(total_rev / 1e9, 2) AS total_rev_bn
    ORDER BY avg_rating DESC
    LIMIT 15
""", label="Q5 — Top Cinematographers by Rating")


=== Q5 — Top Cinematographers by Rating ===
         cinematographer  films  avg_rating  total_rev_bn
          Laurent Chalet      5        8.06          0.00
Dimas Bagus Triatma Yoga      6        7.92          0.01
            Atsushi Okui      8        7.87          1.37
          Kazuto Izumita      5        7.78          0.02
         Tristan Whitman      5        7.74          0.00
             Chase Smith      7        7.71          0.00
             Steve Organ      5        7.70          0.00
             Rory Taylor      6        7.60          0.00
        Mathieu Giombini      7        7.57          0.00
          Joan Churchill      5        7.56          0.00
        Björn Henriksson      9        7.49          0.00
           Trent Opaloch      7        7.48          7.32
               Eiji Arai     10        7.48          0.01
          Dereck Joubert      7        7.44          0.00
          Daniel B. Gold      6        7.43          0.00


,cinematographer,films,avg_rating,total_rev_bn
0,Laurent Chalet,5,8.06,0.00
1,Dimas Bagus Triatma Yoga,6,7.92,0.01
2,Atsushi Okui,8,7.87,1.37
3,Kazuto Izumita,5,7.78,0.02
4,Tristan Whitman,5,7.74,0.00
5,Chase Smith,7,7.71,0.00
6,Steve Organ,5,7.70,0.00
7,Rory Taylor,6,7.60,0.00
8,Mathieu Giombini,7,7.57,0.00
9,Joan Churchill,5,7.56,0.00


In [19]:
# Q6 — Top Composers by Portfolio
run("""
    MATCH (m:Movie)-[:SCORE_BY]->(c:Composer)
    WITH c, COUNT(m) AS films, AVG(m.vote_average) AS avg_rating, SUM(m.revenue) AS total_rev
    WHERE films >= 5
    RETURN c.name AS composer, films,
           ROUND(avg_rating, 2) AS avg_rating,
           ROUND(total_rev / 1e9, 2) AS total_rev_bn
    ORDER BY films DESC
    LIMIT 15
""", label="Q6 — Top Composers by Portfolio")


=== Q6 — Top Composers by Portfolio ===
           composer  films  avg_rating  total_rev_bn
          Koji Endo    159        4.20          0.42
  Alexandre Desplat    137        6.21         10.07
 Yuvan Shankar Raja    135        5.46          0.12
         Ippei Yogo    121        1.62          0.00
        A.R. Rahman    113        6.32          1.54
    Christophe Beck     96        5.98          8.67
     Giuseppe Verdi     93        2.75          0.00
      Goro Yasukawa     92        5.48          0.03
        John Debney     86        6.23          8.01
          S. Thaman     84        5.37          0.19
      Bruno Coulais     82        5.81          0.54
        Kenji Kawai     82        6.24          1.00
        Roque Baños     81        6.05          0.92
G. V. Prakash Kumar     80        5.84          0.13
         Mark Isham     76        6.54          2.68


,composer,films,avg_rating,total_rev_bn
0,Koji Endo,159,4.20,0.42
1,Alexandre Desplat,137,6.21,10.07
2,Yuvan Shankar Raja,135,5.46,0.12
3,Ippei Yogo,121,1.62,0.00
4,A.R. Rahman,113,6.32,1.54
5,Christophe Beck,96,5.98,8.67
6,Giuseppe Verdi,93,2.75,0.00
7,Goro Yasukawa,92,5.48,0.03
8,John Debney,86,6.23,8.01
9,S. Thaman,84,5.37,0.19


In [20]:
# Q7 — Power Director-Actor Combos (collaboration frequency + avg rating)
run("""
    MATCH (d:Director)<-[:DIRECTED_BY]-(m:Movie)-[:ACTED_IN]->(a:Actor)
    WITH d, a, COUNT(m) AS collabs, AVG(m.vote_average) AS avg_rating
    WHERE collabs >= 2
    RETURN d.name AS director, a.name AS actor, collabs,
           ROUND(avg_rating, 2) AS avg_rating
    ORDER BY collabs DESC, avg_rating DESC
    LIMIT 20
""", label="Q7 — Power Director-Actor Collaborations")


=== Q7 — Power Director-Actor Collaborations ===
      director                 actor  collabs  avg_rating
    Kevin Dunn             John Cena      178        7.29
    Kevin Dunn          Glenn Jacobs      161        7.19
    Kevin Dunn           Randy Orton      160        7.24
    Kevin Dunn            Paul Wight      152        7.15
    Kevin Dunn         Paul Levesque      149        7.24
    Kevin Dunn          Chris Irvine      147        7.22
    Kevin Dunn         Adam Copeland      142        7.31
    Kevin Dunn          Mark Calaway      132        7.30
Hiroyuki Tsuji         Hitoshi Ozawa      115        0.29
    Kevin Dunn  Michael Hickenbottom      114        7.35
    Kevin Dunn          Mike Mizanin      107        7.19
    Kevin Dunn         Matthew Hardy      107        7.15
Hiroyuki Tsuji     Yasukaze Motomiya      103        0.40
    Kevin Dunn            Jeff Hardy      102        6.98
    Kevin Dunn            Kurt Angle      100        7.15
    Kevin Dunn Óscar G

,director,actor,collabs,avg_rating
0,Kevin Dunn,John Cena,178,7.29
1,Kevin Dunn,Glenn Jacobs,161,7.19
2,Kevin Dunn,Randy Orton,160,7.24
3,Kevin Dunn,Paul Wight,152,7.15
4,Kevin Dunn,Paul Levesque,149,7.24
5,Kevin Dunn,Chris Irvine,147,7.22
6,Kevin Dunn,Adam Copeland,142,7.31
7,Kevin Dunn,Mark Calaway,132,7.30
8,Hiroyuki Tsuji,Hitoshi Ozawa,115,0.29
9,Kevin Dunn,Michael Hickenbottom,114,7.35


In [24]:
# Q7.1 — Power Director-Actor Combos (collaboration frequency + avg rating) (Excluding Wrestlemania)
run("""
    MATCH (d:Director)<-[:DIRECTED_BY]-(m:Movie)-[:ACTED_IN]->(a:Actor)
    WHERE m.vote_count >= 50 
      AND m.runtime <= 300
      AND NOT (
          toLower(m.overview) CONTAINS 'wrestlemania'
          OR toLower(m.overview) CONTAINS 'wrestling'
          OR toLower(m.overview) CONTAINS 'wwe'
          OR toLower(m.overview) CONTAINS 'wwf'
          OR toLower(m.overview) CONTAINS 'smackdown'
      )
    WITH d, a, COUNT(m) AS collabs, AVG(m.vote_average) AS avg_rating
    WHERE collabs >= 2
    RETURN d.name AS director, a.name AS actor, collabs,
           ROUND(avg_rating, 2) AS avg_rating
    ORDER BY collabs DESC, avg_rating DESC
    LIMIT 20
""", label="Q7 — Power Director-Actor Collaborations")

C:\Users\OM\AppData\Local\Temp\ipykernel_22324\2769207873.py:13: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as s:



=== Q7 — Power Director-Actor Collaborations ===
            director                actor  collabs  avg_rating
     Kunihiko Yuyama   Megumi Hayashibara       19        6.67
     Kunihiko Yuyama       Unsho Ishizuka       19        6.67
     Kunihiko Yuyama      Koichi Yamadera       19        6.67
     Kunihiko Yuyama     Shin-ichiro Miki       19        6.67
     Kunihiko Yuyama       Rica Matsumoto       19        6.67
     Kunihiko Yuyama           Ikue Otani       19        6.67
         Tyler Perry          Tyler Perry       19        6.34
     Kunihiko Yuyama        Inuko Inuyama       18        6.69
        Neri Parenti    Christian De Sica       17        4.81
     Kunihiko Yuyama            Yuji Ueda       15        6.74
        Ridley Scott       Giannina Facio       12        6.70
     Kunihiko Yuyama       Shoko Nakagawa       11        6.74
          Johnnie To             Lam Suet       11        6.74
     Santiago Segura      Santiago Segura       11        6.29
  M. 

,director,actor,collabs,avg_rating
0,Kunihiko Yuyama,Megumi Hayashibara,19,6.67
1,Kunihiko Yuyama,Unsho Ishizuka,19,6.67
2,Kunihiko Yuyama,Koichi Yamadera,19,6.67
3,Kunihiko Yuyama,Shin-ichiro Miki,19,6.67
4,Kunihiko Yuyama,Rica Matsumoto,19,6.67
5,Kunihiko Yuyama,Ikue Otani,19,6.67
6,Tyler Perry,Tyler Perry,19,6.34
7,Kunihiko Yuyama,Inuko Inuyama,18,6.69
8,Neri Parenti,Christian De Sica,17,4.81
9,Kunihiko Yuyama,Yuji Ueda,15,6.74


In [21]:
# Q8 — Director + Producer Power Pairs
run("""
    MATCH (d:Director)<-[:DIRECTED_BY]-(m:Movie)-[:PRODUCED_BY]->(p:Producer)
    WHERE m.revenue IS NOT NULL
    WITH d, p, COUNT(m) AS collabs, AVG(m.roi_pct) AS avg_roi, SUM(m.revenue) AS total_rev
    WHERE collabs >= 2
    RETURN d.name AS director, p.name AS producer, collabs,
           ROUND(avg_roi, 1) AS avg_roi_pct,
           ROUND(total_rev / 1e9, 2) AS total_rev_bn
    ORDER BY total_rev DESC
    LIMIT 15
""", label="Q8 — Director-Producer Power Pairs")


=== Q8 — Director-Producer Power Pairs ===
         director          producer  collabs  avg_roi_pct  total_rev_bn
    James Cameron     James Cameron        4        515.7          6.78
    James Cameron        Jon Landau        3        677.3          6.77
    Peter Jackson     Peter Jackson       10        511.2          6.53
    Peter Jackson        Fran Walsh        8        511.2          6.51
      David Yates     Lionel Wigram        7        390.7          6.04
      David Yates      David Heyman        7        390.7          6.04
Christopher Nolan       Emma Thomas        9        341.4          6.02
Christopher Nolan Christopher Nolan        8        365.3          5.65
      Michael Bay       Michael Bay       10        234.1          5.20
      Michael Bay         Ian Bryce        8        260.2          4.68
     Ridley Scott      Ridley Scott       19        125.4          4.56
      David Yates      David Barron        5        430.4          4.52
      Michael Bay   

,director,producer,collabs,avg_roi_pct,total_rev_bn
0,James Cameron,James Cameron,4,515.7,6.78
1,James Cameron,Jon Landau,3,677.3,6.77
2,Peter Jackson,Peter Jackson,10,511.2,6.53
3,Peter Jackson,Fran Walsh,8,511.2,6.51
4,David Yates,Lionel Wigram,7,390.7,6.04
5,David Yates,David Heyman,7,390.7,6.04
6,Christopher Nolan,Emma Thomas,9,341.4,6.02
7,Christopher Nolan,Christopher Nolan,8,365.3,5.65
8,Michael Bay,Michael Bay,10,234.1,5.20
9,Michael Bay,Ian Bryce,8,260.2,4.68


In [22]:
# Q9 — Golden Trio: Director + Writer + Composer same film, avg success
run("""
    MATCH (d:Director)<-[:DIRECTED_BY]-(m:Movie)-[:WRITTEN_BY]->(w:Writer)
    MATCH (m)-[:SCORE_BY]->(c:Composer)
    WITH d, w, c, COUNT(m) AS shared_films, AVG(m.vote_average) AS avg_rating
    WHERE shared_films >= 2
    RETURN d.name AS director, w.name AS writer, c.name AS composer,
           shared_films, ROUND(avg_rating, 2) AS avg_rating
    ORDER BY avg_rating DESC
    LIMIT 15
""", label="Q9 — Director + Writer + Composer Golden Trios")


=== Q9 — Director + Writer + Composer Golden Trios ===
                       director            writer                        composer  shared_films  avg_rating
Jonathan Kent, Jonathan Haswell  Giuseppe Giacosa                 Giacomo Puccini             2        9.50
Jonathan Kent, Jonathan Haswell      Luigi Illica                 Giacomo Puccini             2        9.50
                 Dominik Sedlar    Dominik Sedlar              Dalibor Grubačević             2        9.45
                   Donovan Cook   Mark Seidenberg Mike Himelstein, Michael Turner             2        9.25
                  Mark Linfield     Mark Linfield                   Nitin Sawhney             2        9.00
                   Theo Anthony      Theo Anthony                      Dan Deacon             2        8.50
              Kazutaka Watanabe    Hirohiko Araki               Naruyoshi Kikuchi             2        8.45
              Kazutaka Watanabe  Yasuko Kobayashi               Naruyoshi Kikuch

,director,writer,composer,shared_films,avg_rating
0,"Jonathan Kent, Jonathan Haswell",Giuseppe Giacosa,Giacomo Puccini,2,9.50
1,"Jonathan Kent, Jonathan Haswell",Luigi Illica,Giacomo Puccini,2,9.50
2,Dominik Sedlar,Dominik Sedlar,Dalibor Grubačević,2,9.45
3,Donovan Cook,Mark Seidenberg,"Mike Himelstein, Michael Turner",2,9.25
4,Mark Linfield,Mark Linfield,Nitin Sawhney,2,9.00
5,Theo Anthony,Theo Anthony,Dan Deacon,2,8.50
6,Kazutaka Watanabe,Hirohiko Araki,Naruyoshi Kikuchi,2,8.45
7,Kazutaka Watanabe,Yasuko Kobayashi,Naruyoshi Kikuchi,2,8.45
8,Narthan,Narthan,Ravi Basrur,2,8.30
9,Raj B Shetty,Raj B Shetty,Midhun Mukundan,3,8.20


In [23]:
# Q10 — Most Versatile Actors: distinct genres covered
run("""
    MATCH (a:Actor)<-[:ACTED_IN]-(m:Movie)
    WHERE m.genres IS NOT NULL
    WITH a, COUNT(m) AS total_films, COLLECT(DISTINCT m.genres) AS genre_list
    WHERE total_films >= 10
    RETURN a.name AS actor, total_films,
           SIZE(genre_list) AS unique_genre_combos
    ORDER BY unique_genre_combos DESC
    LIMIT 15
""", label="Q10 — Most Versatile Actors by Genre Coverage")

driver.close()


=== Q10 — Most Versatile Actors by Genre Coverage ===
                   actor  total_films  unique_genre_combos
            Eric Roberts          356                  170
            Frank Welker          219                  159
            Grey DeLisle          189                  154
       Dee Bradley Baker          176                  136
             Danny Trejo          208                  126
                  Nassar          291                  123
             Tara Strong          163                  123
             Prakash Raj          290                  122
Kevin Michael Richardson          144                  121
               Tom Kenny          162                  116
         Fred Tatasciore          154                  114
         Koichi Yamadera          147                  111
            Brahmanandam          334                  109
           Tomokazu Seki          139                  105
           Lochlyn Munro          146                  100
